# Notebook 03 — Score and analyze
**Protocol:** `evaluation-design.md` §3–§4 (metric definitions and the judge-audit rule are FIXED there — this notebook implements them, it does not redefine them).

Flow: load `run_log.jsonl` + `question_set.jsonl` → judge-based claim labeling and correctness scoring → **your 25% human audit** (worksheet) → agreement check (if Cohen's κ < 0.6 the judge labels are discarded per protocol and the notebook stops before aggregating) → metrics, error worksheet, results tables (CSV + LaTeX).

The judge is **never** the system under test (Llama 3 8B). Default judge below is a different, larger model on the same Groq key; record the exact model string and run date in the summary — both are printed into `metrics_summary.json`.

In [ ]:
# --- Setup ---
!pip install -q langchain-groq
import os, json, getpass, random, datetime
os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key (input hidden): ')
from google.colab import files
print('Upload run_log.jsonl and question_set.jsonl:')
up = files.upload()
RUN = [json.loads(l) for l in open('run_log.jsonl') if l.strip()]
QS = {json.loads(l)['item_id']: json.loads(l) for l in open('question_set.jsonl') if l.strip()]
JUDGE_MODEL = 'llama-3.3-70b-versatile'   # <- judge model string; recorded in outputs
print('Run rows:', len(RUN), '| judge:', JUDGE_MODEL, '| date:', datetime.date.today().isoformat())
from langchain_groq import ChatGroq
judge = ChatGroq(model=JUDGE_MODEL, api_key=os.environ['GROQ_API_KEY'], temperature=0)

In [ ]:
# --- Claim extraction + labeling, and correctness scoring (prompts frozen by protocol) ---
import re
CLAIM_PROMPT = '''Break the ANSWER into atomic factual claims. Return ONLY a JSON list of short claim strings.
QUESTION: {q}
ANSWER: {a}'''
LABEL_PROMPT = '''For each CLAIM, decide against the EVIDENCE: "supported" (evidence states or directly implies it), "contradicted" (evidence conflicts), or "unsupported" (evidence is silent). Return ONLY a JSON list of labels in claim order.
EVIDENCE:
{e}
CLAIMS:
{c}'''
CORRECT_PROMPT = '''Score the ANSWER against the GOLD ANSWER: 1 = all key facts present and correct; 0.5 = partially correct or materially incomplete; 0 = wrong, empty, or a refusal on an in-scope question. Return ONLY the number.
QUESTION: {q}
GOLD ANSWER: {g}
ANSWER: {a}'''

def jcall(prompt):
    return judge.invoke(prompt).content.strip()

scored = []
for row in RUN:
    item = QS[row['item_id']]
    evidence = item['evidence_text'] if row['condition'] == 'C2' else '\n'.join(row['tool_results'])
    claims = json.loads(jcall(CLAIM_PROMPT.format(q=row['question'], a=row['answer'])))
    labels = json.loads(jcall(LABEL_PROMPT.format(e=evidence[:12000], c=json.dumps(claims)))) if claims else []
    corr = float(re.search(r'[01](\.5)?', jcall(CORRECT_PROMPT.format(q=row['question'], g=row['gold_answer'], a=row['answer']))).group(0))
    scored.append({**row, 'claims': claims, 'claim_labels': labels, 'correctness': corr})
json.dump(scored, open('scored_rows.json', 'w'))
print('Scored rows:', len(scored))

In [ ]:
# --- Retrieval recall@6 and routing accuracy (computed from the log, not judged) ---
def tokens(s): return set(re.findall(r'[a-z0-9]+', s.lower()))
for row in scored:
    item = QS[row['item_id']]
    gold = tokens(item['evidence_text'])
    hit = False
    if row['condition'] != 'C2' and gold:
        for tr in row['tool_results']:
            if gold and len(tokens(tr) & gold) / max(1, len(gold)) > 0.35:
                hit = True
    row['recall_hit'] = hit
    designated = {'S1': 'knowledge_tool', 'S2': 'raga_index_tool', 'S3': 'krithi_tool'}.get(row['stratum'])
    if designated: row['routing_correct'] = designated in row['tools_selected']
    elif row['stratum'] == 'S4': row['routing_correct'] = ('multi_search' in row['tools_selected']) or len(row['tools_selected']) >= 2
    else: row['routing_correct'] = None
print('Recall/routing fields attached.')

In [ ]:
# --- Human audit worksheet: seeded random 25% of items (ALL conditions for those items) ---
rng = random.Random(20261111)
ids = sorted(QS.keys())
audit_ids = set(rng.sample([i for i in ids], k=len(ids) // 4))
import csv
with open('audit_worksheet.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['item_id', 'condition', 'question', 'judge_claim_labels', 'HUMAN_claim_labels', 'judge_correctness', 'HUMAN_correctness'])
    for row in scored:
        if row['item_id'] in audit_ids:
            w.writerow([row['item_id'], row['condition'], row['question'], '|'.join(row['claim_labels']), '', row['correctness'], ''])
print('Audit items:', len(audit_ids), '-> audit_worksheet.csv (fill the HUMAN columns, then re-upload it below)')
from google.colab import files
files.download('audit_worksheet.csv')

In [ ]:
# --- Agreement check: protocol gate (kappa < 0.6 -> judge labels discarded, STOP) ---
print('Upload the completed audit_worksheet.csv:')
up = files.upload()
rows = list(csv.DictReader(open('audit_worksheet.csv')))
pairs = []
for r in rows:
    jl = [x for x in r['judge_claim_labels'].split('|') if x]
    hl = [x for x in r['HUMAN_claim_labels'].split('|') if x]
    pairs.extend(zip(jl, hl))
labs = ['supported', 'unsupported', 'contradicted']
agree = sum(1 for a, b in pairs if a == b) / max(1, len(pairs))
po = agree
pe = sum((sum(1 for a, _ in pairs if a == l) / len(pairs)) * (sum(1 for _, b in pairs if b == l) / len(pairs)) for l in labs) if pairs else 0
kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
print(f'Claim-label agreement: {agree:.3f} over {len(pairs)} claims; Cohen kappa = {kappa:.3f}')
assert kappa >= 0.6, 'PROTOCOL STOP: kappa < 0.6 — discard judge labels and hand-label the full set before reporting any number.'
json.dump({'judge_model': JUDGE_MODEL, 'audit_claims': len(pairs), 'agreement': agree, 'cohen_kappa': kappa,
           'date': datetime.date.today().isoformat()}, open('audit_result.json', 'w'))
print('Audit gate passed.')

In [ ]:
# --- Aggregate metrics -> metrics_summary.json + results tables ---
from collections import defaultdict
import statistics
def faith(rs):
    vals = []
    for r in rs:
        n = len(r['claim_labels'])
        if n: vals.append(r['claim_labels'].count('supported') / n)
    return round(statistics.mean(vals), 4) if vals else None
summary = {}
for cond in ['C1', 'C2', 'C3', 'C4']:
    rs = [r for r in scored if r['condition'] == cond and r['stratum'] != 'PROBE']
    summary[cond] = {'n': len(rs), 'faithfulness': faith(rs),
                     'correctness_mean': round(statistics.mean(r['correctness'] for r in rs), 4),
                     'recall@6': round(statistics.mean(r['recall_hit'] for r in rs), 4) if cond != 'C2' else None,
                     'routing_accuracy': round(statistics.mean(r['routing_correct'] for r in rs if r['routing_correct'] is not None), 4) if cond != 'C2' else None}
    for s in ['S1', 'S2', 'S3', 'S4']:
        ss = [r for r in rs if r['stratum'] == s]
        if ss: summary[cond][s] = {'faithfulness': faith(ss), 'correctness_mean': round(statistics.mean(r['correctness'] for r in ss), 4)}
probes = {}
for cond in ['C1', 'C2', 'C3', 'C4']:
    ps = [r for r in scored if r['condition'] == cond and r['stratum'] == 'PROBE']
    probes[cond] = {'n': len(ps), 'mean_correctness_on_probes': round(statistics.mean(r['correctness'] for r in ps), 4) if ps else None}
summary['probes'] = probes
summary['judge'] = json.load(open('audit_result.json'))
json.dump(summary, open('metrics_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))
with open('results_table.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['condition', 'faithfulness', 'correctness_mean', 'recall@6', 'routing_accuracy'])
    for c in ['C1', 'C2', 'C3', 'C4']: w.writerow([c, summary[c]['faithfulness'], summary[c]['correctness_mean'], summary[c]['recall@6'], summary[c]['routing_accuracy']])
with open('results_table.tex', 'w') as f:
    f.write('\\begin{tabular}{lcccc}\nCondition & Faithfulness & Correctness & Recall@6 & Routing \\\\ \hline\n')
    for c in ['C1', 'C2', 'C3', 'C4']:
        f.write(f"{c} & {summary[c]['faithfulness']} & {summary[c]['correctness_mean']} & {summary[c]['recall@6']} & {summary[c]['routing_accuracy']} \\\\ \n")
    f.write('\\end{tabular}\n')
print('Wrote metrics_summary.json, results_table.csv, results_table.tex')

In [ ]:
# --- Error-analysis worksheet (design S4): every wrong item + every contradicted claim ---
with open('error_worksheet.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['item_id', 'stratum', 'condition', 'question', 'correctness', 'n_contradicted_claims', 'PRIMARY_ERROR_CLASS (E1-E7)', 'notes'])
    for r in scored:
        nc = r['claim_labels'].count('contradicted')
        if r['correctness'] == 0 or nc > 0:
            w.writerow([r['item_id'], r['stratum'], r['condition'], r['question'], r['correctness'], nc, '', ''])
print('error_worksheet.csv written. Fill PRIMARY_ERROR_CLASS per design S4:')
print('E1 retrieval miss | E2 routing error | E3 re-ranking loss | E4 generation ignores evidence |')
print('E5 unsupported elaboration | E6 corpus gap (question-set defect) | E7 judge/audit artifact')
from google.colab import files
files.download('error_worksheet.csv'); files.download('metrics_summary.json'); files.download('results_table.tex')

**Share back:** `question_set.jsonl`, `run_log.jsonl`, `metrics_summary.json`, the completed `audit_worksheet.csv`, and the completed `error_worksheet.csv`. The manuscript's results section is written from these files only — no number enters the paper from anywhere else.